# Lab 3 — one controlled change and a defensible recommendation

100 points · 6% NP · same pairs as Lab 2 · due Thu 29 October 2026, 12:30 Santiago.

Lab 1 checked data and simple rules. Lab 2 changed one input under fixed comparison conditions.
Here you keep the inputs fixed and change one model setting, then explain errors and write for a
Head of Strategy. The supplied code runs the experiment; you must supply the checked explanation,
one-change specification, three supported failure patterns, memo and truthful AI record.

Follow the numbered headings. See README for setup and GLOSSARY for terms.
A losing additional candidate can earn full credit. The unchanged starter is incomplete.


## Environment and data check

Extract the whole ZIP and choose your kernel before running this block. Follow README.md if a check fails.

In [1]:
from pathlib import Path
import sys
import importlib
print('Selected Python:', sys.executable)
print('Version:', sys.version.split()[0], '| working directory:', Path.cwd())
required_files = ['lab03_support_v3.py', 'pitwindow_student_v2.py', 'data/learning_v2.csv']
missing = [name for name in required_files if not Path(name).is_file()]
if missing:
    raise FileNotFoundError('Extract the complete ZIP and open its folder as cwd. Missing: ' + ', '.join(missing))
for module in ['numpy', 'pandas', 'sklearn', 'scipy', 'joblib']:
    loaded = importlib.import_module(module)
    print(module, loaded.__version__)
print('Ready: local files/imports found. Next cell checks data integrity and split.')


Selected Python: C:\Users\UDD\Documents\UDD\Cursos_2026_2\IA_Workshop_2026_03\IIT414W_T3_2026\tmp\l3v2env\Scripts\python.exe
Version: 3.12.4 | working directory: C:\Users\UDD\Documents\UDD\Cursos_2026_2\IA_Workshop_2026_03\IIT414W_T3_2026\02_diseno_evaluacion\Lab3_v3\qa\fresh extraction with spaces 4
numpy 2.0.2
pandas 2.3.1
sklearn 1.8.0
scipy 1.17.1
joblib 1.4.2
Ready: local files/imports found. Next cell checks data integrity and split.


In [2]:
from pathlib import Path
import pandas as pd
from lab03_support_v3 import (
    FEATURES, SEED, BUDGET, load_learning, run_base_comparison,
    run_one_change, export_run, markdown_table,
)
ROOT = Path('.')
train, dev = load_learning(ROOT)
print('Seed:', SEED, '| development alert budget:', BUDGET)
print('Permitted inputs:', FEATURES)
audit = pd.DataFrame([
    {'split': name, 'rows': len(df), 'positive_windows': int(df.y.sum()),
     'eligible_events': df.loc[df.y.eq(1), 'event_id'].nunique(),
     'first_year': int(df.year.min()), 'last_year': int(df.year.max())}
    for name, df in [('train', train), ('dev', dev)]
])
display(audit)
assert set(train.opportunity_id).isdisjoint(dev.opportunity_id)


,split,rows,positive_windows,eligible_events,first_year,last_year
0,train,63165,5523,1934,2018,2021
1,dev,9451,1073,370,2022,2022


Seed: 414 | development alert budget: 0.1
Permitted inputs: ['circuit_id', 'compound', 'tyre_life', 'lap_number', 'progress', 'prior_pit_entries', 'pace_median_s', 'pace_slope_s_per_lap', 'pace_n']


## 1. Define the prediction

**Your answer:**

- **Prediction time.** One opportunity (window) is created at the end of a rival driver's lap L (`cut_s`, race time at that lap end). The model may only use what is recorded up to that moment.
- **Positive label.** `y = 1` if that same rival is observed entering the pit lane during their own next three laps (L+1 … L+3); otherwise `y = 0`. It is an *observed* entry by the rival, not an optimal strategy call for our own car. Several consecutive windows can point to the same entry (`event_id`), which is why we count both windows and distinct events. In the data, every positive window has `event_s > cut_s`, so the entry is always after the prediction time.
- **One allowed input.** `tyre_life`: the age in laps of the tyre set the rival is on at the end of lap L. It is known at the prediction time (it is last reported and can lag by a lap, as GLOSSARY notes). It is one of the nine `FEATURES` passed to the pipeline.
- **One future-known field kept out of X.** `event_s` (and the related `event_id`) records *when* the pit entry happened. It exists only after the entry, so it is used only to audit the label and count events; it never enters `FEATURES`. If it were an input, the model would effectively read the answer (leakage). `horizon_end_s` and `future_neutralisation` are also future-known and excluded.
- **What this retrospective snapshot can and cannot establish.** It is a corrected historical reconstruction (FastF1 timing, `manifest_lab03_v3.json`). It can show that, as recorded afterwards, each input describes the state at or before lap L, and that train (2018–2021) precedes development (2022, rounds 1–11). It cannot establish that these values would arrive on time in a live feed (latency, later corrections or missing laps, e.g. `pace_median_s` is missing in about 2% of rows). It also cannot show how the alert would behave in races after round 11 of 2022 or in a prospective trial.

## Practice reading the counts

Illustrative manual example only, NOT F1 data. Two positive windows refer to event A; a third refers
to event B. The final window is negative and triggers a false alert.

| Window | y | Event | Alert |
|---|---:|---|---|
| w1 | 1 | A | yes |
| w2 | 1 | A | yes |
| w3 | 1 | B | no |
| w4 | 0 | none | yes |

Ungraded check: count TP, FP, FN, alerts, eligible/caught events, precision and event recall.
What changes if w2 stops alerting? Think first, then open the self-check below.


### Optional practice response

Before opening the self-check below, write or think through: TP = __; FP = __; FN = __; alerts = __; eligible/caught events = __ / __; precision = __; event recall = __. If w2 becomes quiet: __. This exercise is ungraded; no separate submission is required.

<details>
<summary>Open after trying: self-check for the four fictional windows</summary>

TP=2, FP=1, FN=1, TN=0; alerts=3; precision=2/3. There are two eligible events A/B, one caught, so event recall=1/2. If w2 becomes quiet: TP=1, FP=1, FN=2, alerts=2, precision=1/2; A is still caught by w1 and event recall remains 1/2.

</details>

In [3]:
base_comparison, fitted, base_policies = run_base_comparison(train, dev)
compact_columns = ['model', 'alerts', 'precision', 'events_caught', 'eligible_events', 'event_recall']
print('Start with DEVELOPMENT. Undefined precision means there were no alerts.')
display(base_comparison.loc[base_comparison.split.eq('dev'), compact_columns])
columns = ['model', 'split', 'n', 'TP', 'FP', 'FN', 'TN', 'alerts', 'alert_rate',
           'precision', 'events_caught', 'eligible_events', 'event_recall', 'threshold']
print("Audit view: train and development reuse the threshold selected for that model on dev.")
display(base_comparison[columns])


,model,alerts,precision,events_caught,eligible_events,event_recall
1,never_alert,0,NaN,0,370,0.000000
3,historical_baseline,931,0.268528,94,370,0.254054
5,tree,571,0.269702,63,370,0.170270
7,forest_all,938,0.276119,107,370,0.289189
9,forest_sqrt,945,0.359788,140,370,0.378378


,model,split,n,TP,FP,FN,TN,alerts,alert_rate,precision,events_caught,eligible_events,event_recall,threshold
0,never_alert,train,63165,0,0,5523,57642,0,0.000000,NaN,0,1934,0.000000,NaN
1,never_alert,dev,9451,0,0,1073,8378,0,0.000000,NaN,0,370,0.000000,NaN
2,historical_baseline,train,63165,3042,6645,2481,50997,9687,0.153360,0.314029,1137,1934,0.587901,0.161926
3,historical_baseline,dev,9451,250,681,823,7697,931,0.098508,0.268528,94,370,0.254054,0.161926
4,tree,train,63165,1816,4831,3707,52811,6647,0.105232,0.273206,650,1934,0.336091,0.713735
5,tree,dev,9451,154,417,919,7961,571,0.060417,0.269702,63,370,0.170270,0.713735
6,forest_all,train,63165,2503,4503,3020,53139,7006,0.110916,0.357265,969,1934,0.501034,0.718176
7,forest_all,dev,9451,259,679,814,7699,938,0.099249,0.276119,107,370,0.289189,0.718176
8,forest_sqrt,train,63165,2671,6414,2852,51228,9085,0.143830,0.294001,1005,1934,0.519648,0.608283
9,forest_sqrt,dev,9451,340,605,733,7773,945,0.099989,0.359788,140,370,0.378378,0.608283


Start with DEVELOPMENT. Undefined precision means there were no alerts.
Audit view: train and development reuse the threshold selected for that model on dev.


## 2. Explain the common comparison

**Development first (9,451 windows, 1,073 positive windows, 370 eligible pit entries; budget = at most 945 alerts).**

| System | Alerts (rate) | TP / alerts = precision | Events caught / eligible = event recall | Threshold |
|---|---:|---:|---:|---:|
| never_alert | 0 (0%) | 0 / 0 = undefined | 0 / 370 = 0.000 | none |
| historical_baseline | 931 (9.9%) | 250 / 931 = 0.269 | 94 / 370 = 0.254 | 0.162 |
| tree | 571 (6.0%) | 154 / 571 = 0.270 | 63 / 370 = 0.170 | 0.714 |
| forest_all | 938 (9.9%) | 259 / 938 = 0.276 | 107 / 370 = 0.289 | 0.718 |
| forest_sqrt | 945 (10.0%) | 340 / 945 = 0.360 | 140 / 370 = 0.378 | 0.608 |

**forest_all vs forest_sqrt.** With almost the same number of alerts (938 vs 945, +7), forest_sqrt catches 33 more distinct entries (140 vs 107) and issues fewer false alerts (605 vs 679 FP). It is the only learned model clearly above the historical baseline (94/370). The two forests are identical except `max_features`. After preprocessing, each tree sees 48 columns (checkable with `len(fitted['forest_sqrt'][:-1].get_feature_names_out())`): 7 numeric inputs, plus missing-value indicators and the one-hot columns for `circuit_id` and `compound`. forest_all (`max_features=1.0`) may choose among all 48 at every split. forest_sqrt (`'sqrt'`) draws a fresh random set of int(√48) = 6 candidate columns at each split. No column is removed: every column remains available at other splits and in other trees. Our hypothesis, not measured here: when all 48 are always available, the strongest numeric inputs tend to win the same top splits in most trees, so the 150 trees look alike. Restricting the candidates forces other splits and makes the averaged score rank windows differently. We did not measure tree similarity, so this explains a plausible mechanism, not a proven cause.

**Reading the counts.**
- *Undefined precision:* never_alert issues 0 alerts, so precision is 0/0 = undefined, not 0. Its event recall is 0/370.
- *TP windows are not caught events:* forest_sqrt has 340 TP windows but 140 caught entries, about 2.4 alerted positive windows per caught entry. The 370 entries have 1,073 positive windows, because one pit entry is inside the 3-lap horizon of up to three consecutive laps. Event recall counts each entry once if *any* of its windows alerts.
- *Same policy, different thresholds:* each model uses the same rule on dev: at most 10% alerts, maximise distinct entries caught, with precision then threshold as tie-breakers. Scores are on different scales, so the numeric cutoffs differ (0.608 vs 0.718). Ties also matter: the depth-3 tree produces only 8 distinct scores. Its next lower score level would alert 1,898 windows, over the 945 limit, so it stops at 571 and leaves budget unused. Comparing systems compares *model + selected threshold*, not the model alone.

**One train/dev gap (forest_sqrt).** Event recall is 1,005/1,934 = 0.520 on train vs 140/370 = 0.378 on dev. This gap is not a clean measure of overfitting:
1. Train scores are in-sample (the forest was fit on those windows).
2. The dev-selected threshold, reused unchanged on train, alerts 14.4% of train windows (9,085), because the 10% budget applies only to dev. More alerts mechanically catch more entries.
3. Precision moves the other way (train 0.294 vs dev 0.360), which a simple "memorised train" story would not predict.

Dev itself was used to choose thresholds and to compare systems, so its numbers are also somewhat optimistic and are not an untouched final test. The gap alone cannot tell us its cause (season change, in-sample fit or the different alert rate).

## 3. Record your expectation before your additional run

The base outputs are already supplied. Complete `experiment_spec.md` before running YOUR candidate
for the first time. If already run, say so and keep a retrospective account. Do not invent history.

Choose ONE tuple: tree / max_depth / 6 (old 3); forest_sqrt / max_depth / 3 (old 6);
or forest_sqrt / min_samples_leaf / 100 (old 30). No search for the best of all three.
The helper clones the base pipeline and changes one setting. Its threshold is reselected on dev
under the same policy, so the numeric threshold may differ. This compares the resulting alert systems.


Choose one additional configuration. Repeating that same configuration for debugging or Restart + Run All is allowed. Disclose any different configurations you already tried; do not hide a search.

In [4]:
EXPERIMENT_MODEL_ID = None  # 'tree' or 'forest_sqrt', matching the allowed table
EXPERIMENT_PARAMETER = None  # 'max_depth' or 'min_samples_leaf'
EXPERIMENT_VALUE = None      # the one allowed new value


In [5]:
change = None
policies = dict(base_policies)
comparison = base_comparison.copy()
if any(value is None for value in [EXPERIMENT_MODEL_ID, EXPERIMENT_PARAMETER, EXPERIMENT_VALUE]):
    print('INCOMPLETE STARTER: configure one additional experiment in extra-config.')
else:
    extra_rows, extra_model, extra_policy, change = run_one_change(
        train, dev, fitted, EXPERIMENT_MODEL_ID, EXPERIMENT_PARAMETER, EXPERIMENT_VALUE)
    comparison = pd.concat([base_comparison, extra_rows], ignore_index=True)
    policies[change['model_id']] = extra_policy
    display(comparison.loc[comparison.model.isin([change['parent'], change['model_id']]), columns])


INCOMPLETE STARTER: configure one additional experiment in extra-config.


## 4. Explain your additional result

Replace this prompt with your answer. Compare the candidate with its unchanged parent using at least
two actual numbers. Did the recorded expectation hold? Explain the parameter's plausible mechanism,
separating observed changes from causes you did not measure. Include a bounded recommendation and
the limits of dev selection and historical timing. Keep the original expectation in the spec.
Consider both the fitted model and its separately selected alert threshold. No calibrated-probability or racing-gain claims.


## 5. Explain three distinct failure patterns

Keep your additional candidate and its selected threshold frozen. Inspect the supplied tables, choose the denominator for your claim, and use the queries to find supporting cases. This step does not train or retune.

### Choose your denominator

| Question within your context | Calculation | Denominator counts |
|---|---|---|
| Of the alerts issued, how many were false? | FP / (TP + FP) | Alerted windows; false-alert fraction, equal to 1 − precision when defined. |
| Of the negative windows, how many alerted? | FP / (FP + TN) | Negative windows; false positive rate (FPR). |
| Of the positive windows, how many were quiet? | FN / (TP + FN) | Positive windows; `miss_rate` in the context export. |
| Of the eligible events, how many were completely missed? | Completely missed events / eligible events | Distinct pit entries; use all positive windows for each event. |

A zero denominator means undefined, shown as NaN; it does not mean zero errors. Name your population and unit with every fraction. Window context groups do not define an event population automatically: if describing missed events in a context, state which distinct events you include and check all their positive windows, including those outside that context. The event query below checks a whole event.


### A complete entry from the four fictional windows

This uses only the four-window practice example, not the Rival dataset or an additional candidate's results.

“For the context of all four toy windows, w4 is a false alert (y=0, alert=yes). One of the three alerts is false: FP/alerts=1/3. The traceable case is `opportunity_id=w4`. This could prompt an unnecessary review of a rival that does not enter within the horizon; next check the time and outcome around w4 before attributing a cause. With only four invented windows, this illustrates the reasoning format and cannot establish a general effect.”

The same counts give FPR=1/(1+0)=1/1 over negative windows, and miss_rate=1/(2+1)=1/3 over positive windows. Those fractions answer different questions. Event B has one positive window (w3), none alerted: one of two eligible events is completely missed. A quiet w2 alone would not make event A missed because w1 still alerts.


In [6]:
from lab03_support_v3 import development_predictions, context_error_summary, event_error_summary
predictions = None
if change is None:
    print('INCOMPLETE: run your additional candidate before examining its errors.')
else:
    predictions = development_predictions(dev, extra_model, extra_policy, change['model_id'])
    context_errors = context_error_summary(predictions)
    event_errors = event_error_summary(predictions)
    print('Descriptive context tables. No refit or threshold change occurs here.')
    display(context_errors)
    print('FP window examples (highest scores first):')
    display(predictions.loc[predictions.error_type.eq('FP')].sort_values('score', ascending=False).head(8))
    print('FN window examples (lowest scores first):')
    display(predictions.loc[predictions.error_type.eq('FN')].sort_values('score').head(8))
    print('Missed events:', int(event_errors.missed_event.sum()), '/', len(event_errors))
    display(event_errors.loc[event_errors.missed_event].head(8))


INCOMPLETE: run your additional candidate before examining its errors.


### Find the cases behind your claim

Set CONTEXT_COLUMN to a supplied context column (for example compound or circuit_id), CONTEXT_VALUE to an actual value of the same type shown in your predictions, and ERROR_KIND to FP or FN. The summary uses all rows in that context, so its denominators do not accidentally count only errors. `cases` contains every matching case; the display previews eight. For three patterns you can repeat the query with recorded settings, or copy this query block within this section. Keep each setting and relevant output alongside its written claim.

For a whole-event claim, copy an event_id from the missed-event table into EVENT_TO_INSPECT. The second query retrieves all its positive windows, including their opportunity_id values, without a context filter. An event is missed only if none alerts. Unknown IDs return no rows and are not evidence of a missed event.


In [7]:
from lab03_support_v3 import context_query, positive_windows_for_event

CONTEXT_COLUMN = None
CONTEXT_VALUE = None
ERROR_KIND = 'FP'

if predictions is None:
    print('Run your additional candidate first; its frozen predictions are needed.')
elif CONTEXT_COLUMN is None or CONTEXT_VALUE is None:
    print('Set the context column/value above to inspect your own error evidence.')
else:
    context_summary, cases = context_query(predictions, CONTEXT_COLUMN, CONTEXT_VALUE, ERROR_KIND)
    display(context_summary)
    print('Matching cases:', len(cases), '(first eight shown)')
    display(cases.head(8))


Run your additional candidate first; its frozen predictions are needed.


In [8]:
EVENT_TO_INSPECT = None

if predictions is None:
    print('Run your additional candidate first.')
elif EVENT_TO_INSPECT is None:
    print('Set an event_id above when checking a completely missed event.')
else:
    event_windows = positive_windows_for_event(predictions, EVENT_TO_INSPECT)
    display(event_windows)
    if event_windows.empty:
        print('No positive windows found: check the ID; do not call this a missed event.')
    else:
        print('Positive windows:', len(event_windows), 'Alerted:', int(event_windows.alert.sum()))
        print('Completely missed event:', not bool(event_windows.alert.any()))


Run your additional candidate first.


### What counts as a separate pattern

The same error type in different contexts can count as separate patterns when each entry states its context rule, gives its own count/denominator and traceable case, and explains a consequence and next check. Three cases of the same error in the same context count as one pattern. Changing only a context name without evidence and interpretation is incomplete. You do not need three different causes, independent groups or statistical significance. Declare overlap; do not add overlapping counts as distinct errors.

Fictional examples for interpreting the requirement (these are not Rival results):

- Acceptable entry A: “For context `compound == TOY_A`, 2 of 6 alerts are false (FP/alerts), including `toy-A-4`. These alerts require checking a rival that does not enter within the horizon; next check the neighbouring windows to see whether alerts repeat around the same situation. The six-alert sample is small.”
- Acceptable entry B: “For context `compound == TOY_B`, 3 of 10 alerts are false, including `toy-B-2`. The review burden also occurs here; next inspect the recorded timing around those cases before proposing a cause.” With their separate supporting rows, A and B count as two patterns even though both are FP. Your submission must provide the actual rows behind its own claims.
- Incomplete: “TOY_A has false alerts; TOY_B has false alerts; TOY_C has false alerts.” Context names alone supply neither counts/denominators nor cases, consequences or checks. Three IDs from TOY_A under the same rule also describe one pattern.


### Your three entries

Replace the placeholders below using your candidate's fixed development predictions.
For each pattern give an error/context rule, count and denominator, an opportunity_id, consequence
and useful next check. Use about 2–3 short sentences per pattern, plus the numbers/ID. Three rows
with the same error/context are one pattern. Patterns may overlap: state it; do not add counts as independent.

1. Pattern / rule: ... Evidence count / denominator: ... opportunity_id: ... Consequence / next check: ...
2. Pattern / rule: ... Evidence count / denominator: ... opportunity_id: ... Consequence / next check: ...
3. Pattern / rule: ... Evidence count / denominator: ... opportunity_id: ... Consequence / next check: ...

The context table is window-level. If claiming an entirely missed entry, verify it in event_errors,
not from one FN window. Explain whether each proposed cause is measured or a hypothesis. No retuning
or new model is needed. If evidence is insufficient, record what you checked and the limitation; never invent patterns.


In [9]:
manifest = export_run(comparison, policies, change, ROOT, predictions=predictions)
print('Additional experiment executed:', manifest['experiment_complete'])
print('This flag does NOT check your reasoning, three patterns, memo or AI record.')
print('Systems:', comparison.model.nunique(), '| train/dev rows:', len(comparison))
print('Generated comparison_table.md and all five outputs/ files. No manual transcription needed.')


Additional experiment executed: False
This flag does NOT check your reasoning, three patterns, memo or AI record.
Systems: 5 | train/dev rows: 10
Generated comparison_table.md and all five outputs/ files. No manual transcription needed.


## 6. Memo and submission

Complete memo.md and PROMPTS.md. The comparison table is generated automatically.
Check CHECKLIST.md, restart + run all, save visible outputs and push the final commit.
Completed comparison: six systems / twelve train-dev rows, plus your additional candidate's diagnostics.
Canvas text entry: both names, accessible repository URL, final commit and notebook path.
The execution flag does not certify your explanation or completed documents.


## Use your feedback next time

Ungraded preparation: revisit one relevant Lab 2 comment, or identify a weakness yourself if no
comment applies. What did you check differently here? Add one or two sentences if useful.
After grading, carry the specific next action from your Lab 3 feedback into the capstone experiment plan.
No extra file, points or oral interview is added by this reflection.
